In [1]:
# generate_car_env.py

# Assuming ast_config.py and gym_generator.py are in the same directory.
# You might need to adjust imports if they are in different modules/packages.

from ast_config import (
    GymEnvironmentConfig, DefinitionsConfig, VariablesConfig, ControlFuncConfig,
    StepFuncConfig, ProblemConfig,
    VariableConfig, ConstantDefinitionConfig, FunctionDefinitionConfig, PredicateDefinitionConfig,
    NondeterministicAssignmentActionConfig, DeterministicAssignmentActionConfig, ActionConstraintConfig,
    SequenceActionConfig, ChoiceActionConfig,
    ODEConfig, EvolutionDomainConfig,
    PreconditionConfig, PostconditionConfig, LoopExecutionConfig,
    CallHybridProgramConfig,
    ActionType # <--- THIS IS THE CRUCIAL IMPORT YOU NEED TO ADD
)
from gym_generator import generate_gym_environment_code # Import the generator function

# --- 1. Define Definitions (Constants, Functions, Predicates) ---
# No special constants, functions, or predicates for this simple car example.
definitions_config = DefinitionsConfig(
    constants=[],
    functions=[],
    predicates=[]
)

# --- 2. Define Variables (State and Auxiliary) ---
state_variables = [
    VariableConfig(name="x", type="real", lower_bound="0.0", upper_bound="100.0"), # Position
    VariableConfig(name="v", type="real", lower_bound="-10.0", upper_bound="10.0"), # Velocity
]
auxiliary_variables = [
    # Define 'a' as an auxiliary variable with its bounds.
    # This helps the generator infer the action space correctly.
    VariableConfig(name="a", type="real", lower_bound="-1.0", upper_bound="1.0"), # Acceleration action
]

variables_config = VariablesConfig(
    state_variables=state_variables,
    auxiliary_variables=auxiliary_variables
)

# --- 3. Define Control Function (Action Space) ---
# The agent chooses an acceleration 'a' between -1.0 and 1.0
control_func_config = ControlFuncConfig(
    name="car_control",
    body_raw="{a := *;}", # Added required body_raw argument
    action_structure=SequenceActionConfig(
        actions=[
            NondeterministicAssignmentActionConfig(variable="a"), # Agent chooses 'a'
            # The bounds for 'a' are now explicitly defined in auxiliary_variables.
            # The generator should use these to create a Box space.
        ]
    )
)

# --- 4. Define Step Function (Environment Dynamics) ---
# ODEs: dx/dt = v, dv/dt = a
odes = [
    ODEConfig(variable="x", equation="v"),
    ODEConfig(variable="v", equation="a")
]
# Evolution domain: always true (ODEs always run)
# Changed from 'formula' to 'constraints: List[str]'
evolution_domain = EvolutionDomainConfig(constraints=["True"])

step_func_config = StepFuncConfig(
    name="car_dynamics", # Name is now a required argument for HybridProgramConfig
    body_raw="x'=v, v'=a & True", # Added required body_raw argument
    odes=odes,
    evolution_domain=evolution_domain, # Non-default argument comes first
    duration="0.1" # Each step integrates for 0.1 seconds
)

# Add hybrid programs to definitions_config
definitions_config.hybrid_programs.append(control_func_config)
definitions_config.hybrid_programs.append(step_func_config)


# --- 5. Define Problem Configuration ---
# Precondition: x=0, v=0. We'll use a formula that's true if these conditions are met.
# The generator's reset method will initialize state variables.
# For this example, we'll assume the initial values set in reset() satisfy the precondition.
precondition = PreconditionConfig(formula="x == 0 and v == 0")

# Postcondition: Goal (x >= 100) or off track (x < 0).
# Let's make it a boolean expression for termination and reward.
# The generator will interpret 'True' as goal reached for reward, 'False' otherwise.
postcondition = PostconditionConfig(formula="x >= 100 or x < 0")

# Loop execution: 1 step function execution per Gymnasium step
# The loop body should explicitly call stepFunc for continuous dynamics.
loop_execution = LoopExecutionConfig(
    iterations="1",
    loop_body=SequenceActionConfig(
        actions=[
            CallHybridProgramConfig(hp_name="car_dynamics") # Call the stepFunc
        ]
    )
)

problem_config = ProblemConfig(
    # Removed 'name="CarProblem"' as ProblemConfig does not have a name field
    precondition=precondition,
    loop_execution=loop_execution, # Corrected field name
    postcondition=postcondition
)

# --- Assemble the full GymEnvironmentConfig ---
car_env_config = GymEnvironmentConfig(
    env_name="CarEnv",
    definitions=definitions_config,
    variables=variables_config,
    problem=problem_config
    # Removed control_func and step_func as direct arguments; they are now in definitions.hybrid_programs
)

# --- 6. Generate the Environment Code ---
generated_code = generate_gym_environment_code(car_env_config)

# --- 7. Save the Generated Code to a File ---
output_filename = "generated_car_env.py"
with open(output_filename, "w") as f:
    f.write(generated_code)

print(f"Generated Gymnasium environment code saved to {output_filename}")

# --- 8. How to Use the Generated Environment (after saving and running this script) ---
print("\nTo use the generated environment, run the following in a separate Python script or interpreter:")
print(f"""
import gymnasium as gym
from {output_filename.replace('.py', '')} import CarEnv
import numpy as np

# Register the environment (optional, but good practice for gym.make)
# gym.register(
#     id='CarEnv-v0',
#     entry_point='{output_filename.replace('.py', '')}:CarEnv',
#     max_episode_steps=200, # Example max steps
# )

# env = gym.make('CarEnv-v0') # If registered
env = CarEnv() # If not registered, instantiate directly

observation, info = env.reset()
print("Initial observation:", observation)
print("Observation Space:", env.observation_space)
print("Action Space:", env.action_space)

# Example action: apply an acceleration of 0.5
# The action space is a Dict, so the action should be a dictionary.
# For NondeterministicAssignmentActionConfig(variable="a"), the key will be 'assign_a'.
action = {{'assign_a': 0.5}} # Corrected action to be a dictionary

for i in range(10): # Take 10 steps
    observation, reward, terminated, truncated, info = env.step(action)
    # Ensure observation elements are accessed correctly, e.g., observation[0] for x, observation[1] for v
    # Assuming observation is a numpy array or tuple of (x, v)
    # Corrected escaping: {{...}} for expressions in the *inner* f-string
    print(f"Step {{i+1}}: Obs=(x={{observation[0]:.2f}}, v={{observation[1]:.2f}}), Reward={{reward:.2f}}), Terminated={{terminated}}), Truncated={{truncated}})")
    if terminated or truncated:
        print("Episode finished.")
        break

env.close()
""")

Generated Gymnasium environment code saved to generated_car_env.py

To use the generated environment, run the following in a separate Python script or interpreter:

import gymnasium as gym
from generated_car_env import CarEnv
import numpy as np

# Register the environment (optional, but good practice for gym.make)
# gym.register(
#     id='CarEnv-v0',
#     entry_point='generated_car_env:CarEnv',
#     max_episode_steps=200, # Example max steps
# )

# env = gym.make('CarEnv-v0') # If registered
env = CarEnv() # If not registered, instantiate directly

observation, info = env.reset()
print("Initial observation:", observation)
print("Observation Space:", env.observation_space)
print("Action Space:", env.action_space)

# Example action: apply an acceleration of 0.5
# The action space is a Dict, so the action should be a dictionary.
# For NondeterministicAssignmentActionConfig(variable="a"), the key will be 'assign_a'.
action = {'assign_a': 0.5} # Corrected action to be a dictionary

for i